# RoomRubiksPack Interactive Example (Client-Server Split)

This notebook demonstrates the complete workflow for generating an architectural floorplan layout using the `roomrubikspack` client library.

In this split architecture model, your client registers layouts and handles visualisation and exports, while a RoomRubiks backend server runs the Elitist Genetic Algorithm (GA) computations.

### 0. Install the Package & Dependencies

We ensure the package is installed along with the optional dependencies required for plotting (`matplotlib`, `networkx`), DXF export (`ezdxf`), and HTTP calls (`requests`).

In [ ]:
!pip install -e ..
!pip install matplotlib networkx ezdxf requests

### 1. Initialization and Server Setup (`rr.init`, `rr.settings`)

We import the package and initialize the session. `rr.init()` clears any previous state. We also configure the server endpoint. By default, it looks for `http://127.0.0.1:8000` (localhost).

In [ ]:
import sys
import os

# Ensure Jupyter knows where to find the source code
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

import roomrubikspack as rr

# Clear session state
rr.init()

# Set the server URL pointing to your API server
rr.settings(unit="m", server_url="http://127.0.0.1:8000")

### 2. Grid Control (`rr.constructiongrid`)

We can manipulate the underlying grid module sizes used to generate room dimensions.

In [ ]:
# View or modify the construction grid
rr.constructiongrid(add=5.0)
rr.constructiongrid(remove=9.0)
rr.constructiongrid(reset=True)  # Reset back to defaults
rr.constructiongrid()  # Prints the current grid

### 3. Defining Rooms and Site (`rr.room` and `rr.site`)

Define the rooms by providing explicitly known dimensions (`w` and `h`), or just by providing a target `area` (in square meters). We can also optionally provide a site boundary.

**Important Flags:**
- `startSpace=True`: Indicates the entry room or central hub. Exactly **one** room must have this flag.
- `attachedSpace=True`: Indicates this room is 'dependent' and will be placed *inside* or physically grouped tightly with its connected parent.

In [ ]:
rr.room("living",   "Living Room",  area=20.0, startSpace=True, color="#f2e6d9")
rr.room("kitchen",  "Kitchen",      w=3.0, h=3.0, color="#d9f2d9")
rr.room("bed1",     "Master Bed",   area=16.0, color="#d9d9f2")
rr.room("bath1",    "Attached Bath", area=4.0, attachedSpace=True, color="#e6f2ff")

# (Optional) Define a site boundary
rr.site([{"x": -2, "y": -2}, {"x": 22, "y": -2}, {"x": 22, "y": 22}, {"x": -2, "y": 22}])

### 4. Establishing Connectivity (`rr.connectivity`)

Define which rooms must be physically adjacent to each other. When this runs, it performs a local Euler planarity check to warn you if the graph is impossible to lay out without crossing.

In [ ]:
rr.connectivity(
    ("living", "kitchen"),
    ("living", "bed1"),
    ("bed1",   "bath1")
)

### 5. Visualizing the Graph (`rr.connectivityshow`)

Plots the adjacency graph locally. The start space will be highlighted in green.

In [ ]:
rr.connectivityshow()

### 6. Adding Soft Constraints (`rr.constraint`)

Constraints are sent to the server to guide the Genetic Algorithm. Supported constraints:
- `position`: Bias a room toward a cardinal direction.
- `area`: Attempt to optimize the total building footprint towards a target area.
- `perimeter`: Attempt to minimize the external bounding box perimeter.

In [ ]:
rr.constraint("position", "bed1", "N")
rr.constraint("area", None, 120)
rr.constraint("perimeter", None, "minimize")

### 7. Auto-generating Dimensions (`rr.dimensiongen`)

Sends the rooms to the server to calculate standard architectural dimensions based on target areas.

In [ ]:
rr.dimensiongen(avar=0.10, mar=1.5)

### 8. Layout Generation (`rr.generatelayout`)

Sends all session data, connectivity, and constraints to the RoomRubiks backend server to run layout optimization.

In [ ]:
rr.generatelayout(lvar=0.5, sgap=1.0, max_variations=5)

### 9. Visualizing Layouts Locally (`rr.showlayout`)

Plots the calculated layout variation locally using Matplotlib.

In [ ]:
rr.showlayout(n=1, label=["name", "id", "dim", "area"])

### 10. Exporting Layouts Locally (`rr.exportlayout`)

Saves the layout variation locally as JSON or CAD-compatible DXF format.

In [ ]:
rr.exportlayout(n=1, filepath="layout_var1.json")
rr.exportlayout(n=1, filepath="layout_var1.dxf")